# CIC IIoT Dataset 2025 Intrusion Detection - GWO 

In [ ]:
!pip install scikit-learn
!pip install mealpy
!pip install Boruta
!pip install lightgbm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from typing import List, Tuple, Dict, Optional
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import VarianceThreshold
_HAS_MEALPY = False
_HAS_BORUTA = False
try:
    from boruta import BorutaPy
    _HAS_BORUTA = True
    print('BorutaPy available')
except Exception:
    _HAS_BORUTA = False
    print('BorutaPy not available yet. Run the install cell, then re-run this cell.')
try:
    from mealpy.swarm_based import GWO, BA
    _HAS_MEALPY = True
    print("Mealpy available: GWO/BA")
except Exception:
    _HAS_MEALPY = False
    print("Mealpy not available. Using fallback feature selection.")

_HAS_FLOATVAR = False
try:
    from mealpy import FloatVar
    _HAS_FLOATVAR = True
except Exception:
    _HAS_FLOATVAR = False

plt.style.use('seaborn-v0_8-darkgrid')

BorutaPy available
Mealpy available: GWO/BA


## Feature Selection Functions (GWO + Original BorutaPy + Optional BA Helpers + Without FS)

In [ ]:
def _evaluate_subset(
    X: np.ndarray, y: np.ndarray, idx: List[int],
    clf=None, cv: int = 3, scoring: str = 'accuracy',
    lam: float = 0.01,
    alpha_fp: float = 0.0,   # set >0 later if you want explicit false-positive penalty
    benign_id: Optional[int] = None
) -> float:
    if len(idx) == 0:
        return 0.0

    # CHANGED: default classifier now class-weighted for robustness
    if clf is None:
        clf = DecisionTreeClassifier(random_state=42, class_weight='balanced')

    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler(with_mean=True)),
        ('clf', clf)
    ])

    X_sub = X[:, idx]
    cvk = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

    scorer = 'accuracy' if scoring == 'accuracy' else make_scorer(f1_score, average='macro')

    # Base CV score
    scores = cross_val_score(pipe, X_sub, y, scoring=scorer, cv=cvk, n_jobs=None)
    base = float(np.mean(scores))

    # Compactness penalty
    size_penalty = lam * (len(idx) / X.shape[1])

    # Optional false-positive penalty (OFF by default)
    fp_penalty = 0.0
    if alpha_fp > 0.0 and benign_id is not None:
        fprs = []
        for tr, te in cvk.split(X_sub, y):
            pipe.fit(X_sub[tr], y[tr])
            p = pipe.predict(X_sub[te])
            y_te = y[te]
            benign = (y_te == benign_id)
            fp = np.sum((p != benign_id) & benign)
            tn = np.sum((p == benign_id) & benign)
            fprs.append(fp / (fp + tn + 1e-12))
        fp_penalty = alpha_fp * float(np.mean(fprs))

    return base - size_penalty - fp_penalty

# =========================================
# Convergence History Storage (Global)
# =========================================
convergence_history = {}

def _optimize_continuous(
    OptimizerClass, X: np.ndarray, y: np.ndarray, k: int,
    clf=None, epochs: int = 30, pop_size: int = 30,
    cv: int = 3, scoring: str = 'accuracy', seed: int = 42
) -> Tuple[List[int], float]:

    dim = X.shape[1]
    lb_vec = [0.0] * dim
    ub_vec = [1.0] * dim

    lambda_max = 0.05
    lambda_min = 0.005

    total_calls = epochs * pop_size
    call_counter = {'n': 0}

    history = []

    def fitness(w):
        call_counter['n'] += 1
        progress = min(call_counter['n'] / total_calls, 1.0)
        t = progress * epochs

        a_t = 2 - 2 * (t / epochs)
        lam_dynamic = lambda_min + (lambda_max - lambda_min) * (a_t / 2)

        w = np.asarray(w)
        idx = list(range(dim)) if k >= dim else list(np.argsort(-w)[:k])

        score = _evaluate_subset(
            X, y, idx,
            clf=clf,
            cv=cv,
            scoring=scoring,
            lam=lam_dynamic
        )

        history.append(score)
        return score

    random.seed(int(seed))
    np.random.seed(int(seed))

    problem_mealpy = {
        'obj_func': fitness,
        'minmax': 'max',
        'lb': lb_vec,
        'ub': ub_vec,
        'seed': int(seed)
    }

    if _HAS_FLOATVAR:
        problem_mealpy['bounds'] = FloatVar(lb=lb_vec, ub=ub_vec)

    if isinstance(OptimizerClass, type):
        opt = OptimizerClass(epoch=epochs, pop_size=pop_size)
        res = opt.solve(problem_mealpy)
        algo_name = OptimizerClass.__name__
    else:
        if hasattr(OptimizerClass, 'OriginalGWO'):
            opt = OptimizerClass.OriginalGWO(epoch=epochs, pop_size=pop_size)
            algo_name = "GWO"
        elif hasattr(OptimizerClass, 'OriginalBA'):
            opt = OptimizerClass.OriginalBA(epoch=epochs, pop_size=pop_size)
            algo_name = "BA"
        else:
            raise TypeError('Unsupported optimizer type')
        res = opt.solve(problem_mealpy)

    best_pos = res[0] if isinstance(res, tuple) else res.solution
    # Save convergence history
    convergence_history[f"{algo_name}_k{k}"] = history

    w = np.asarray(best_pos)
    idx = list(range(dim)) if k >= dim else list(np.argsort(-w)[:k])

    final_lambda = lambda_min
    score = _evaluate_subset(
        X, y, idx,
        clf=clf,
        cv=cv,
        scoring=scoring,
        lam=final_lambda
    )

    return idx, score


def select_features_gwo(X, y, k, clf=None, epochs=30, pop_size=30, cv=3, scoring='accuracy', seed: int = 42):
    if _HAS_MEALPY:
        return _optimize_continuous(GWO, X, y, k, clf, epochs, pop_size, cv, scoring, seed=seed)

    from sklearn.feature_selection import mutual_info_classif
    mi = mutual_info_classif(X, y, random_state=42)
    idx = list(np.argsort(-mi)[:k])

    # CHANGED: apply penalty in fallback too
    return idx, _evaluate_subset(X, y, idx, clf=clf, cv=cv, scoring=scoring, lam=0.01)


def select_features_ba(X, y, k, clf=None, epochs=30, pop_size=30, cv=3, scoring='accuracy'):
    if _HAS_MEALPY:
        return _optimize_continuous(BA, X, y, k, clf, epochs, pop_size, cv, scoring)

    from sklearn.feature_selection import mutual_info_classif
    mi = mutual_info_classif(X, y, random_state=42)
    idx = list(np.argsort(-mi)[:k])

    # CHANGED: apply penalty in fallback too
    return idx, _evaluate_subset(X, y, idx, clf=clf, cv=cv, scoring=scoring, lam=0.01)


def select_features_boruta(X, y, k=None, clf=None, cv=3, scoring='accuracy', seed: int = 42):
    if not _HAS_BORUTA:
        raise ImportError("BorutaPy is not available. Install Boruta, then rerun the imports cell.")

    X_imp = SimpleImputer(strategy='median').fit_transform(X)
    y_arr = np.asarray(y)
    dim = int(X_imp.shape[1])

    rf = RandomForestClassifier(
   n_estimators=800,
    max_depth=30,
    min_samples_split=3,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced',
    max_samples=0.8,
    n_jobs=-1,
    random_state=42
    )

    boruta = BorutaPy(
        estimator=rf,
        n_estimators='auto',
        perc=100,
        alpha=0.05,
        two_step=True,
        max_iter=15,
        random_state=int(seed),
        verbose=0
    )
    boruta.fit(X_imp, y_arr)

    idx = list(np.where(np.asarray(boruta.support_, dtype=bool))[0])
    if len(idx) == 0:
        ranking = np.asarray(getattr(boruta, 'ranking_', np.ones(dim)), dtype=float)
        idx = [int(np.argmin(ranking))]

    return idx, _evaluate_subset(X, y_arr, idx, clf=clf, cv=cv, scoring=scoring, lam=0.0)


def select_features_hybrid_boruta(X, y, k, clf=None, cv=3, scoring='accuracy', seed: int = 42):
    X_imp = SimpleImputer(strategy='median').fit_transform(X)
    y_arr = np.asarray(y)
    dim = int(X_imp.shape[1])
    kk = int(min(max(int(k), 1), dim))

    rng = np.random.default_rng(int(seed))
    n_iter = int(max(10, int(cv) * 3))
    wins = np.zeros(dim, dtype=float)
    imp_sum = np.zeros(dim, dtype=float)

    for _ in range(n_iter):
        X_shadow = X_imp.copy()
        for j in range(dim):
            rng.shuffle(X_shadow[:, j])
        X_aug = np.hstack([X_imp, X_shadow])

        rf = RandomForestClassifier(
       n_estimators=800,
    max_depth=30,
    min_samples_split=3,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced',
    max_samples=0.8,
    n_jobs=-1,
    random_state=42
        )
        rf.fit(X_aug, y_arr)
        imp = np.asarray(getattr(rf, 'feature_importances_', None), dtype=float)
        if imp is None or imp.size != (2 * dim):
            raise RuntimeError('rf importances not available')

        real_imp = imp[:dim]
        shadow_imp = imp[dim:]
        thr = float(np.max(shadow_imp))
        wins += (real_imp > thr).astype(float)
        imp_sum += real_imp

    score = wins + 1e-3 * (imp_sum / float(n_iter))
    idx = list(np.argsort(-score)[:kk])
    return idx, _evaluate_subset(X, y_arr, idx, clf=clf, cv=cv, scoring=scoring, lam=0.01)


def select_features_without_fs(X, y, clf=None, cv=3, scoring='accuracy'):
    idx = list(range(X.shape[1]))
    return idx, _evaluate_subset(X, y, idx, clf=clf, cv=cv, scoring=scoring, lam=0.0)


def select_features_lightgbm(X, y, k, clf=None, cv=3, scoring='accuracy', seed: int = 42):
    try:
        from lightgbm import LGBMClassifier
    except ImportError:
        raise ImportError('lightgbm is not available.')
    X_imp = SimpleImputer(strategy='median').fit_transform(X)
    y_arr = np.asarray(y)
    kk = int(min(max(int(k), 1), X_imp.shape[1]))
    n_classes = len(np.unique(y_arr))
    objective = 'binary' if n_classes <= 2 else 'multiclass'
    model = LGBMClassifier(
        n_estimators=800,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        objective=objective,
        class_weight='balanced',
        random_state=int(seed),
        verbosity=-1
    )
    model.fit(X_imp, y_arr)
    imp = np.asarray(getattr(model, 'feature_importances_', np.zeros(X_imp.shape[1])), dtype=float)
    if imp.size != X_imp.shape[1] or np.all(imp == 0):
        raise RuntimeError('LightGBM importances not available')
    idx = list(np.argsort(-imp)[:kk])
    return idx, _evaluate_subset(X, y_arr, idx, clf=clf, cv=cv, scoring=scoring, lam=0.01)


def run_all_algorithms(X, y, ks: List[int], clf=None, epochs=30, pop_size=30, cv=3, feature_names: Optional[List[str]] = None, scoring='accuracy') -> Dict[str, Dict[int, Dict]]:
    results = {'GWO': {}, 'BA': {}, 'LightGBM': {}, 'Boruta': {}}
    for k in ks:
        idx_gwo, s_gwo = select_features_gwo(X, y, k, clf, epochs, pop_size, cv, scoring)
        idx_ba, s_ba = select_features_ba(X, y, k, clf, epochs, pop_size, cv, scoring)
        idx_lgbm, s_lgbm = select_features_lightgbm(X, y, k, clf, cv=cv, scoring=scoring)
        idx_boruta, s_boruta = select_features_boruta(X, y, None, clf, cv=cv, scoring=scoring, seed=42)

        for algo, idx, score in [('GWO', idx_gwo, s_gwo), ('BA', idx_ba, s_ba), ('LightGBM', idx_lgbm, s_lgbm), ('Boruta', idx_boruta, s_boruta)]:
            feats = [feature_names[i] for i in idx] if feature_names else None
            results[algo][k] = {'indices': idx, 'score': score, 'features': feats}

    return results

## Load Data + Cleaning + Dimensionality Reduction

In [ ]:
print('Loading and preprocessing dataset...')
all_data = r"/root/iiot-run/data/cleaned_merged_full.csv"

print('Loading data...')
combined_data = pd.read_csv(all_data, on_bad_lines='skip')
#combined_data = pd.read_csv(all_data, on_bad_lines='skip').sample(n=3000, random_state=42)

print(f"Dataset loaded successfully!")
print(f"   - Total samples: {len(combined_data)}")
print(f"   - Features: {combined_data.shape[1] - 1}")

# -------- Detect label column (label2 preferred, else label1)
label_col_src = None
for col in combined_data.columns:
    cl = col.lower()
    if 'label2' in cl:
        label_col_src = col
        break
    if 'label1' in cl and label_col_src is None:
        label_col_src = col

if label_col_src:
    label_percent = (combined_data[label_col_src].value_counts(normalize=True) * 100).round(2)
    print('\nLabel distribution (%):')
    print(label_percent)

# -------- Drop non-numeric string columns but keep label columns
string_columns = combined_data.select_dtypes(include=['object']).columns.tolist()
for keep in ['label2', 'label1', label_col_src]:
    if keep in string_columns:
        string_columns.remove(keep)

print(f"   - Dropping {len(string_columns)} string columns for ML processing")
combined_data_numeric = combined_data.drop(columns=string_columns)

target_col = 'label2' if 'label2' in combined_data_numeric.columns else ('label1' if 'label1' in combined_data_numeric.columns else None)
if target_col is None:
    raise ValueError('No label2/label1 column found in dataset')

X_raw = combined_data_numeric.drop(target_col, axis=1)
y_raw = combined_data_numeric[target_col]

# -------- Multiclass encoding
y_clean = y_raw.astype(str).str.strip()
classes = sorted(y_clean.unique())
class_to_id = {c: i for i, c in enumerate(classes)}
y = y_clean.map(class_to_id).astype(int)
print(f'Encoding used (multiclass): {class_to_id}')

# -------- Fill numeric NaNs with column mean
numeric_columns = X_raw.select_dtypes(include=[np.number]).columns
X_raw[numeric_columns] = X_raw[numeric_columns].fillna(X_raw[numeric_columns].mean())

# -------- Build df for cleaning
df = X_raw.copy()
df['label'] = y
label_col = 'label'
print(f'Dataset: {df.shape[0]} rows, {df.shape[1]} cols')
print('Target column:', label_col)

# -------- Robust time/order split (use Timestamp if present; else use row order)
use_time_split = ('Timestamp' in df.columns)

if use_time_split:
    df['__ts'] = pd.to_datetime(df['Timestamp'], errors='coerce')
    df = df.sort_values('__ts')
else:
    df = df.reset_index(drop=True)

# -------- Drop identifiers (keep Timestamp only for sorting; do not use it as a feature)
drop_cols = ['Protocol', 'Flow_ID', 'Src_IP', 'Dst_IP', 'Src_Port', 'Dst_Port', 'Timestamp']
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

# -------- Feature matrix and target
X_df = df.drop(columns=[label_col]).copy()
y = df[label_col].values

# -------- Convert to numeric + clean inf/nan
for col in X_df.columns:
    X_df[col] = pd.to_numeric(X_df[col], errors='coerce')

X_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# -------- Fix special columns if present
for special in ['Flow_Duration', 'Fwd_Header_Length']:
    if special in X_df.columns:
        bad = (X_df[special] <= 0) | pd.isna(X_df[special])
        med = X_df[special][X_df[special] > 0].median()
        X_df.loc[bad, special] = med if not np.isnan(med) else 0

# -------- Drop columns with too many NaNs
missing_ratio = X_df.isna().mean()
X_df = X_df.loc[:, missing_ratio <= 0.4]

# -------- Fill remaining NaNs
X_df = X_df.apply(lambda c: c.fillna(c.median()))
X_df = X_df.fillna(0)

# -------- Remove zero-variance features
vt = VarianceThreshold(threshold=0.0)
vt.fit(X_df.values)
X_df = X_df.iloc[:, vt.get_support(indices=True)]

# -------- Final valid rows
valid = ~pd.isna(y)
X_df, y = X_df.loc[valid], y[valid]

feature_names = list(X_df.columns)

# -------- Split: train first 80%, test last 20% (robust generalisation test)
# split_point = int(len(X_df) * 0.8)
# X_train_df = X_df.iloc[:split_point].copy()
# X_test_df  = X_df.iloc[split_point:].copy()
# y_train = y[:split_point]
# y_test  = y[split_point:]

#new
from sklearn.model_selection import train_test_split

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)




X = X_train_df.values.astype(float)
y = y_train
X_test = X_test_df.values.astype(float)

print(f'After cleaning:')
print(f'   - Train: {X.shape[0]} rows, {X.shape[1]} columns')
print(f'   - Test:  {X_test.shape[0]} rows, {X_test.shape[1]} columns')
print(f'   - NaNs in X_train: {np.isnan(X).sum()}')
print(f'   - NaNs in X_test:  {np.isnan(X_test).sum()}')

print(np.bincount(y_train))
print(np.bincount(y_test))


Loading and preprocessing dataset...
Loading data...


Dataset loaded successfully!
   - Total samples: 685671
   - Features: 93

Label distribution (%):
label2
benign        58.44
recon         15.44
dos            8.42
ddos           8.27
mitm           3.72
malware        3.53
web            1.32
bruteforce     0.88
Name: proportion, dtype: float64
   - Dropping 21 string columns for ML processing


/tmp/ipykernel_15863/4105004487.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = combined_data.select_dtypes(include=['object']).columns.tolist()


Encoding used (multiclass): {'benign': 0, 'bruteforce': 1, 'ddos': 2, 'dos': 3, 'malware': 4, 'mitm': 5, 'recon': 6, 'web': 7}


Dataset: 685671 rows, 73 cols
Target column: label


After cleaning:
   - Train: 548536 rows, 51 columns
   - Test:  137135 rows, 51 columns
   - NaNs in X_train: 0
   - NaNs in X_test:  0
[320537   4813  45353  46189  19342  20392  84678   7232]
[80135  1203 11339 11547  4835  5098 21170  1808]


## Classification, Feature Count, Time, and Stability Evaluation

In [ ]:
import random
import time
import tracemalloc
from itertools import combinations
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

N_RUNS = 10
RUN_SEEDS = list(range(N_RUNS))
K_SUP = 10
N_FEATURES_TOTAL = int(X.shape[1])

FS_MAX_SAMPLES = 50000
FS_EPOCHS = 30
FS_POP_SIZE = 10
FS_CV_FOLDS = 3

FS_FITNESS_MODEL = 'DecisionTree'
FS_FITNESS_SCORING = 'f1_macro'

EVAL_CV_FOLDS = 3

def _subsample_for_fs_local(X_in: np.ndarray, y_in: np.ndarray, max_samples: int, seed: int):
    if (max_samples is None) or (X_in.shape[0] <= int(max_samples)):
        return X_in, y_in
    rng = np.random.default_rng(int(seed))
    idx = rng.choice(X_in.shape[0], size=int(max_samples), replace=False)
    return X_in[idx], y_in[idx]

def evaluate_cv_eff(clf, X_sub: np.ndarray, y_sub: np.ndarray, cv_folds: int, seed: int):
    cvk = StratifiedKFold(n_splits=int(cv_folds), shuffle=True, random_state=int(seed))

    accs, precs, recs, f1s, mccs = [], [], [], [], []
    train_time_s = 0.0
    infer_time_ms = 0.0

    tracemalloc.start()
    for tr, te in cvk.split(X_sub, y_sub):
        model = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler(with_mean=True)),
            ('clf', clone(clf))
        ])

        t0 = time.perf_counter()
        model.fit(X_sub[tr], y_sub[tr])
        train_time_s += (time.perf_counter() - t0)

        t1 = time.perf_counter()
        p = model.predict(X_sub[te])
        infer_time_ms += (time.perf_counter() - t1) * 1000.0

        accs.append(accuracy_score(y_sub[te], p))
        precs.append(precision_score(y_sub[te], p, average='macro', zero_division=0))
        recs.append(recall_score(y_sub[te], p, average='macro', zero_division=0))
        f1s.append(f1_score(y_sub[te], p, average='macro', zero_division=0))
        mccs.append(matthews_corrcoef(y_sub[te], p))

    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return {
        'accuracy': float(np.mean(accs)),
        'precision_macro': float(np.mean(precs)),
        'recall_macro': float(np.mean(recs)),
        'f1_macro': float(np.mean(f1s)),
        'mcc': float(np.mean(mccs)),
        'train_time_s': float(train_time_s),
        'inference_time_ms': float(infer_time_ms),
        'peak_mem_mb': float(peak) / (1024.0 * 1024.0)
    }

methods = ['GWO', 'BA', 'LightGBM', 'Boruta']

clf_factories = {
    'RandomForest': lambda s: RandomForestClassifier(
        n_estimators=800,
        max_depth=30,
        min_samples_split=3,
        min_samples_leaf=1,
        max_features='sqrt',
        class_weight='balanced',
        max_samples=0.8,
        n_jobs=-1,
        random_state=42
    ),
    'DecisionTree': lambda s: DecisionTreeClassifier(random_state=int(s), class_weight='balanced'),
    'LogisticRegression': lambda s: LogisticRegression(max_iter=1000, class_weight='balanced', random_state=int(s)),
    'GaussianNB': lambda s: GaussianNB(),
    'HistGradientBoosting': lambda s: HistGradientBoostingClassifier(random_state=int(s), learning_rate=0.1),
    'MLP': lambda s: MLPClassifier(
        random_state=int(s),
        hidden_layer_sizes=(128, 64),
        max_iter=200,
        early_stopping=True,
        n_iter_no_change=10,
        validation_fraction=0.1
    )
}

fs_cache = {}

def make_fs_base(seed: int):
    if str(FS_FITNESS_MODEL).lower() in ['rf', 'randomforest', 'random_forest']:
        return RandomForestClassifier(
            n_estimators=120,
            max_depth=18,
            min_samples_split=4,
            min_samples_leaf=2,
            max_features='sqrt',
            bootstrap=True,
            max_samples=0.7,
            class_weight='balanced_subsample',
            n_jobs=-1,
            random_state=int(seed)
        )
    return DecisionTreeClassifier(random_state=int(seed), class_weight='balanced')

def _serialize_idx(idx):
    return ','.join(str(int(i)) for i in idx)

def _mean_pairwise_jaccard(index_sets):
    sets = [set(int(i) for i in s) for s in index_sets if len(s) > 0]
    if len(sets) < 2:
        return np.nan, np.nan, 0
    vals = []
    for a, b in combinations(sets, 2):
        union = len(a | b)
        vals.append(len(a & b) / union if union else 1.0)
    return float(np.mean(vals)), float(np.std(vals)), int(len(vals))

def get_idx_and_time(seed: int, method: str, X_fs: np.ndarray, y_fs: np.ndarray):
    key = (int(seed), str(method))
    if key in fs_cache:
        return fs_cache[key]

    random.seed(int(seed))
    np.random.seed(int(seed))

    base = make_fs_base(int(seed))
    t0 = time.perf_counter()
    if str(method) == 'GWO':
        idx, _ = select_features_gwo(X_fs, y_fs, K_SUP, clf=base, epochs=FS_EPOCHS, pop_size=FS_POP_SIZE, cv=FS_CV_FOLDS, scoring=str(FS_FITNESS_SCORING), seed=int(seed))
    elif str(method) == 'BA':
        idx, _ = select_features_ba(X_fs, y_fs, K_SUP, clf=base, epochs=FS_EPOCHS, pop_size=FS_POP_SIZE, cv=FS_CV_FOLDS, scoring=str(FS_FITNESS_SCORING))
    elif str(method) == 'LightGBM':
        idx, _ = select_features_lightgbm(X_fs, y_fs, K_SUP, clf=base, cv=FS_CV_FOLDS, scoring=str(FS_FITNESS_SCORING), seed=int(seed))
    elif str(method) == 'Boruta':
        idx, _ = select_features_boruta(X_fs, y_fs, None, clf=base, cv=FS_CV_FOLDS, scoring=str(FS_FITNESS_SCORING), seed=int(seed))
    else:
        raise ValueError(f'Unknown method: {method}')

    fs_time_s = float(time.perf_counter() - t0)
    fs_cache[key] = (idx, fs_time_s)
    return idx, fs_time_s

rows = []
for seed in RUN_SEEDS:
    X_fs, y_fs = _subsample_for_fs_local(X, y, FS_MAX_SAMPLES, seed)
    for method in methods:
        idx, fs_time_s = get_idx_and_time(seed, method, X_fs, y_fs)
        X_sub = X[:, idx]
        for clf_name, make_clf in clf_factories.items():
            m = evaluate_cv_eff(make_clf(seed), X_sub, y, cv_folds=EVAL_CV_FOLDS, seed=seed)
            rows.append({
                'seed': int(seed),
                'method': str(method),
                'classifier': str(clf_name),
                'k_target': int(K_SUP) if method in ['GWO', 'BA', 'LightGBM'] else np.nan,
                'n_selected': int(len(idx)),
                'selected_features': _serialize_idx(idx),
                'fs_time_s': float(fs_time_s),
                **m
            })

fs_rows = []
for (s, m), (idx, fs_time_s) in fs_cache.items():
    fs_rows.append({
        'seed': int(s),
        'method': str(m),
        'k_target': int(K_SUP) if str(m) in ['GWO', 'BA', 'LightGBM'] else np.nan,
        'n_selected': int(len(idx)),
        'selected_features': _serialize_idx(idx),
        'fs_time_s': float(fs_time_s)
    })
fs_df = pd.DataFrame(fs_rows)
fs_df = fs_df.sort_values(['seed', 'method']).reset_index(drop=True)
display(fs_df)
display(fs_df.groupby('method', as_index=False)[['fs_time_s', 'n_selected']].agg(['mean', 'std']).reset_index())

stability_rows = []
for method_name, sub_df in fs_df.groupby('method'):
    sets = []
    for txt in sub_df['selected_features'].astype(str):
        sets.append([int(x) for x in txt.split(',') if str(x).strip() != ''])
    mean_j, std_j, n_pairs = _mean_pairwise_jaccard(sets)
    stability_rows.append({
        'method': str(method_name),
        'mean_pairwise_jaccard': mean_j,
        'std_pairwise_jaccard': std_j,
        'n_pairs': n_pairs
    })
stability_df = pd.DataFrame(stability_rows).sort_values('mean_pairwise_jaccard', ascending=False).reset_index(drop=True)
print('Feature-selection stability across runs')
display(stability_df)

raw_df = pd.DataFrame(rows)
display(raw_df)
print('Rows:', len(raw_df))

metrics_cols = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'mcc', 'fs_time_s', 'train_time_s', 'inference_time_ms', 'peak_mem_mb', 'n_selected']

means = raw_df.groupby(['method', 'classifier'])[metrics_cols].mean().reset_index()
means = means.rename(columns={
    'method': 'Algorithm',
    'classifier': 'Classifier',
    'accuracy': 'Accuracy',
    'precision_macro': 'Precision',
    'recall_macro': 'Recall',
    'f1_macro': 'F1_macro',
    'mcc': 'MCC',
    'n_selected': 'N_Selected'
})

eval_df = means[['Algorithm', 'Classifier', 'Accuracy', 'Precision', 'Recall', 'F1_macro', 'MCC', 'N_Selected', 'fs_time_s', 'train_time_s', 'inference_time_ms', 'peak_mem_mb']].copy()
eval_df = eval_df.sort_values(['F1_macro', 'MCC', 'Accuracy'], ascending=[False, False, False]).reset_index(drop=True)
display(eval_df)

print('Deployment-oriented summary (lower downstream cost is better)')
deployment_df = eval_df[['Algorithm', 'Classifier', 'N_Selected', 'train_time_s', 'inference_time_ms', 'peak_mem_mb']].copy()
deployment_df = deployment_df.sort_values(['N_Selected', 'peak_mem_mb', 'inference_time_ms', 'train_time_s'], ascending=[True, True, True, True]).reset_index(drop=True)
display(deployment_df)

agg = raw_df.groupby(['method', 'classifier'])[metrics_cols].agg(['mean', 'std'])
agg.columns = [f"{m}_{s}" for (m, s) in agg.columns]
agg = agg.reset_index()

def fmt_pair(m, s, is_time=False):
    if pd.isna(m):
        return ''
    if pd.isna(s):
        return f"{m:.4f}" if not is_time else f"{m:.3f}"
    return (f"{m:.4f} +- {s:.4f}" if not is_time else f"{m:.3f} +- {s:.3f}")

final_table = pd.DataFrame({
    'Method': agg['method'].astype(str),
    'Classifier': agg['classifier'].astype(str),
})

for src, dst in {
    'accuracy': 'accuracy',
    'precision_macro': 'precision_macro',
    'recall_macro': 'recall_macro',
    'f1_macro': 'f1_macro',
    'mcc': 'mcc',
    'n_selected': 'n_selected',
    'fs_time_s': 'fs_time_s',
    'train_time_s': 'train_time_s',
    'inference_time_ms': 'inference_time_ms',
    'peak_mem_mb': 'peak_mem_mb'
}.items():
    mcol = f"{src}_mean"
    scol = f"{src}_std"
    is_time = src in ['fs_time_s', 'train_time_s', 'inference_time_ms', 'peak_mem_mb']
    final_table[dst] = [fmt_pair(m, s, is_time=is_time) for m, s in zip(agg[mcol], agg[scol])]

final_table = final_table.sort_values(['Classifier', 'Method']).reset_index(drop=True)
display(final_table)

2026/06/08 05:06:45 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: OriginalGWO(epoch=30, pop_size=10)


2026/06/08 05:06:56 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 1, Current best: 0.8342468959884574, Global best: 0.8342468959884574, Runtime: 4.70012 seconds


2026/06/08 05:07:01 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 2, Current best: 0.8443930702359732, Global best: 0.8443930702359732, Runtime: 5.70769 seconds


2026/06/08 05:07:07 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 3, Current best: 0.8529790693294942, Global best: 0.8529790693294942, Runtime: 5.26384 seconds


2026/06/08 05:07:12 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 4, Current best: 0.8529790693294942, Global best: 0.8529790693294942, Runtime: 5.57213 seconds


...

Rows: 300


,Algorithm,Classifier,Accuracy,Precision,Recall,F1_macro,MCC,N_Selected,fs_time_s,train_time_s,inference_time_ms,peak_mem_mb
0,Boruta,RandomForest,0.960053,0.968853,0.931864,0.949486,0.934918,41.2,21.385711,388.908428,38485.466011,632.631626
1,Boruta,DecisionTree,0.962729,0.972628,0.925590,0.947976,0.939333,41.2,21.385711,38.187735,409.809770,632.587744
2,GWO,DecisionTree,0.956968,0.962202,0.915959,0.937937,0.929864,10.0,190.621071,10.910510,162.175361,164.703777
3,LightGBM,RandomForest,0.954305,0.953813,0.920890,0.936615,0.925466,10.0,820.572967,381.883285,40840.455992,259.459781
4,GWO,RandomForest,0.951641,0.949651,0.923434,0.935856,0.921162,10.0,190.621071,329.132580,37471.583837,259.497023
7,Boruta,HistGradientBoosting,0.952378,0.959430,0.905144,0.929961,0.922565,41.2,21.385711,1332.454135,26099.164462,632.648615
8,LightGBM,DecisionTree,0.953985,0.953653,0.906406,0.928889,0.924902,10.0,820.572967,18.496138,170.950099,164.704101
9,GWO,HistGradientBoosting,0.946092,0.958861,0.892748,0.923381,0.912261,10.0,190.621071,1217.765538,24989.439239,164.771596
11,LightGBM,HistGradientBoosting,0.943303,0.949197,0.886039,0.915406,0.907571,10.0,820.572967,1027.472188,23777.845627,164.765708
12,Boruta,MLP,0.943197,0.940609,0.883250,0.909913,0.907212,41.2,21.385711,2190.907379,1709.690491,632.589891


Deployment-oriented summary (lower downstream cost is better)


,Algorithm,Classifier,N_Selected,train_time_s,inference_time_ms,peak_mem_mb
0,BA,GaussianNB,10.0,2.122455,398.912561,164.698232
1,LightGBM,GaussianNB,10.0,2.073779,427.750430,164.698334
3,GWO,GaussianNB,10.0,2.057373,462.896382,164.699038
5,BA,DecisionTree,10.0,10.413739,157.965548,164.702662
6,GWO,DecisionTree,10.0,10.910510,162.175361,164.703777
7,LightGBM,DecisionTree,10.0,18.496138,170.950099,164.704101
9,LightGBM,LogisticRegression,10.0,36.817713,245.679425,164.704584
10,BA,LogisticRegression,10.0,70.537770,322.359367,164.704880
11,GWO,LogisticRegression,10.0,50.744261,303.709876,164.705248
12,BA,HistGradientBoosting,10.0,947.021424,23021.786603,164.764411


,Method,Classifier,accuracy,precision_macro,recall_macro,f1_macro,mcc,n_selected,fs_time_s,train_time_s,inference_time_ms,peak_mem_mb
0,BA,DecisionTree,0.9436 +- 0.0101,0.9249 +- 0.0290,0.8923 +- 0.0150,0.9067 +- 0.0220,0.9080 +- 0.0165,10.0000 +- 0.0000,145.015 +- 10.083,10.414 +- 2.219,157.966 +- 22.250,164.703 +- 0.002
1,Boruta,DecisionTree,0.9627 +- 0.0002,0.9726 +- 0.0006,0.9256 +- 0.0006,0.9480 +- 0.0005,0.9393 +- 0.0003,41.2000 +- 0.6325,21.386 +- 5.048,38.188 +- 1.669,409.810 +- 39.771,632.588 +- 9.485
2,GWO,DecisionTree,0.9570 +- 0.0022,0.9622 +- 0.0048,0.9160 +- 0.0030,0.9379 +- 0.0034,0.9299 +- 0.0036,10.0000 +- 0.0000,190.621 +- 19.284,10.911 +- 1.675,162.175 +- 32.950,164.704 +- 0.002
3,LightGBM,DecisionTree,0.9540 +- 0.0009,0.9537 +- 0.0025,0.9064 +- 0.0025,0.9289 +- 0.0025,0.9249 +- 0.0015,10.0000 +- 0.0000,820.573 +- 499.661,18.496 +- 1.376,170.950 +- 29.374,164.704 +- 0.001
5,BA,GaussianNB,0.7140 +- 0.0577,0.4967 +- 0.0833,0.4880 +- 0.0457,0.4549 +- 0.0693,0.5449 +- 0.0759,10.0000 +- 0.0000,145.015 +- 10.083,2.122 +- 0.398,398.913 +- 75.815,164.698 +- 0.001
6,Boruta,GaussianNB,0.8103 +- 0.0009,0.6298 +- 0.0025,0.6378 +- 0.0022,0.6182 +- 0.0025,0.6893 +- 0.0014,41.2000 +- 0.6325,21.386 +- 5.048,7.962 +- 1.272,1119.407 +- 327.987,632.582 +- 9.484
7,GWO,GaussianNB,0.7535 +- 0.0342,0.5706 +- 0.0447,0.5508 +- 0.0247,0.5292 +- 0.0299,0.6026 +- 0.0411,10.0000 +- 0.0000,190.621 +- 19.284,2.057 +- 0.359,462.896 +- 118.065,164.699 +- 0.003
8,LightGBM,GaussianNB,0.6979 +- 0.0637,0.3955 +- 0.0162,0.3964 +- 0.0068,0.3432 +- 0.0145,0.5145 +- 0.0610,10.0000 +- 0.0000,820.573 +- 499.661,2.074 +- 0.471,427.750 +- 104.662,164.698 +- 0.001
10,BA,HistGradientBoosting,0.9344 +- 0.0063,0.9383 +- 0.0162,0.8643 +- 0.0171,0.8980 +- 0.0160,0.8929 +- 0.0105,10.0000 +- 0.0000,145.015 +- 10.083,947.021 +- 497.669,23021.787 +- 14377.090,164.764 +- 0.005
11,Boruta,HistGradientBoosting,0.9524 +- 0.0038,0.9594 +- 0.0145,0.9051 +- 0.0109,0.9300 +- 0.0130,0.9226 +- 0.0064,41.2000 +- 0.6325,21.386 +- 5.048,1332.454 +- 767.266,26099.164 +- 11370.535,632.649 +- 9.483


## Statistical Significance and Pairwise Tests

In [ ]:
try:
    from scipy.stats import friedmanchisquare, wilcoxon
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

def _prepare_pivot(df: pd.DataFrame, value_col: str, ordered_methods):
    pivot = df.pivot_table(index='seed', columns='method', values=value_col, aggfunc='mean')
    pivot = pivot.dropna(axis=1, how='all')
    ordered = [m for m in list(ordered_methods) if m in pivot.columns]
    pivot = pivot[ordered]
    pivot = pivot.dropna(axis=0, how='any')
    return pivot

def _holm_adjust(pvals):
    if len(pvals) == 0:
        return []
    order = np.argsort(pvals)
    ranked = np.asarray(pvals, dtype=float)[order]
    m = len(ranked)
    adj = np.empty(m, dtype=float)
    running = 0.0
    for i, p in enumerate(ranked):
        value = min(1.0, (m - i) * float(p))
        running = max(running, value)
        adj[i] = running
    restored = np.empty(m, dtype=float)
    restored[order] = adj
    return restored.tolist()

def _safe_wilcoxon(a, b):
    if (not _HAS_SCIPY) or (len(a) < 2):
        return None
    if np.allclose(np.asarray(a) - np.asarray(b), 0.0):
        return 1.0
    return float(wilcoxon(a, b, zero_method='wilcox').pvalue)

def global_significance_for_classifier(df: pd.DataFrame, clf_name: str, methods_list, metric: str = 'f1_macro'):
    sub = df[df['classifier'].astype(str) == str(clf_name)].copy()
    if sub.empty:
        return {'Classifier': str(clf_name), 'metric': metric, 'test': None, 'p_value': None, 'reason': 'no rows'}
    pivot = _prepare_pivot(sub, metric, methods_list)
    methods_avail = list(pivot.columns)
    n_runs = int(pivot.shape[0])
    if len(methods_avail) < 2 or n_runs < 2:
        return {'Classifier': str(clf_name), 'metric': metric, 'test': None, 'p_value': None, 'methods': methods_avail, 'n_runs': n_runs, 'reason': 'need >=2 methods with complete runs'}
    if not _HAS_SCIPY:
        return {'Classifier': str(clf_name), 'metric': metric, 'test': None, 'p_value': None, 'methods': methods_avail, 'n_runs': n_runs, 'reason': 'scipy not available'}
    if len(methods_avail) == 2:
        a, b = methods_avail
        p = _safe_wilcoxon(pivot[a].values, pivot[b].values)
        return {'Classifier': str(clf_name), 'metric': metric, 'test': 'Wilcoxon', 'p_value': p, 'methods': methods_avail, 'n_runs': n_runs}
    samples = [pivot[m].values for m in methods_avail]
    _, p = friedmanchisquare(*samples)
    return {'Classifier': str(clf_name), 'metric': metric, 'test': 'Friedman', 'p_value': float(p), 'methods': methods_avail, 'n_runs': n_runs}

def pairwise_vs_reference_for_classifier(df: pd.DataFrame, clf_name: str, methods_list, reference: str = 'GWO', metric: str = 'f1_macro'):
    sub = df[df['classifier'].astype(str) == str(clf_name)].copy()
    if sub.empty:
        return []
    pivot = _prepare_pivot(sub, metric, methods_list)
    if reference not in pivot.columns:
        return []
    rows = []
    pvals = []
    valid_idx = []
    for method in pivot.columns:
        if str(method) == str(reference):
            continue
        pair = pivot[[reference, method]].dropna(axis=0, how='any')
        n_runs = int(pair.shape[0])
        row = {
            'Classifier': str(clf_name),
            'metric': metric,
            'reference': str(reference),
            'comparison': str(method),
            'n_runs': n_runs,
            'mean_ref': float(pair[reference].mean()) if n_runs else np.nan,
            'mean_cmp': float(pair[method].mean()) if n_runs else np.nan,
            'wins_ref': int((pair[reference] > pair[method]).sum()) if n_runs else 0,
            'ties': int((pair[reference] == pair[method]).sum()) if n_runs else 0,
            'wins_cmp': int((pair[reference] < pair[method]).sum()) if n_runs else 0,
            'test': 'Wilcoxon',
            'p_value': None,
            'p_holm': None,
            'better_by_mean': str(reference) if (n_runs and pair[reference].mean() >= pair[method].mean()) else str(method)
        }
        if _HAS_SCIPY and n_runs >= 2:
            p = _safe_wilcoxon(pair[reference].values, pair[method].values)
            row['p_value'] = p
            if p is not None:
                valid_idx.append(len(rows))
                pvals.append(p)
        else:
            row['test'] = None
        rows.append(row)
    adj = _holm_adjust(pvals)
    for idx, p_adj in zip(valid_idx, adj):
        rows[idx]['p_holm'] = float(p_adj)
    return rows

def global_significance_for_fs(fs_df_local: pd.DataFrame, methods_list, metric: str = 'fs_time_s'):
    pivot = _prepare_pivot(fs_df_local.copy(), metric, methods_list)
    methods_avail = list(pivot.columns)
    n_runs = int(pivot.shape[0])
    if len(methods_avail) < 2 or n_runs < 2:
        return {'metric': metric, 'test': None, 'p_value': None, 'methods': methods_avail, 'n_runs': n_runs, 'reason': 'need >=2 methods with complete runs'}
    if not _HAS_SCIPY:
        return {'metric': metric, 'test': None, 'p_value': None, 'methods': methods_avail, 'n_runs': n_runs, 'reason': 'scipy not available'}
    if len(methods_avail) == 2:
        a, b = methods_avail
        p = _safe_wilcoxon(pivot[a].values, pivot[b].values)
        return {'metric': metric, 'test': 'Wilcoxon', 'p_value': p, 'methods': methods_avail, 'n_runs': n_runs}
    samples = [pivot[m].values for m in methods_avail]
    _, p = friedmanchisquare(*samples)
    return {'metric': metric, 'test': 'Friedman', 'p_value': float(p), 'methods': methods_avail, 'n_runs': n_runs}

def pairwise_vs_reference_for_fs(fs_df_local: pd.DataFrame, methods_list, reference: str = 'GWO', metric: str = 'fs_time_s'):
    pivot = _prepare_pivot(fs_df_local.copy(), metric, methods_list)
    if reference not in pivot.columns:
        return []
    rows = []
    pvals = []
    valid_idx = []
    for method in pivot.columns:
        if str(method) == str(reference):
            continue
        pair = pivot[[reference, method]].dropna(axis=0, how='any')
        n_runs = int(pair.shape[0])
        mean_ref = float(pair[reference].mean()) if n_runs else np.nan
        mean_cmp = float(pair[method].mean()) if n_runs else np.nan
        row = {
            'metric': metric,
            'reference': str(reference),
            'comparison': str(method),
            'n_runs': n_runs,
            'mean_ref': mean_ref,
            'mean_cmp': mean_cmp,
            'wins_ref': int((pair[reference] < pair[method]).sum()) if n_runs else 0,
            'ties': int((pair[reference] == pair[method]).sum()) if n_runs else 0,
            'wins_cmp': int((pair[reference] > pair[method]).sum()) if n_runs else 0,
            'test': 'Wilcoxon',
            'p_value': None,
            'p_holm': None,
            'faster_by_mean': str(reference) if (n_runs and mean_ref <= mean_cmp) else str(method)
        }
        if _HAS_SCIPY and n_runs >= 2:
            p = _safe_wilcoxon(pair[reference].values, pair[method].values)
            row['p_value'] = p
            if p is not None:
                valid_idx.append(len(rows))
                pvals.append(p)
        else:
            row['test'] = None
        rows.append(row)
    adj = _holm_adjust(pvals)
    for idx, p_adj in zip(valid_idx, adj):
        rows[idx]['p_holm'] = float(p_adj)
    return rows

print('Global significance for performance (f1_macro)')
sig_rows = []
for clf_name in sorted(raw_df['classifier'].astype(str).unique()):
    sig_rows.append(global_significance_for_classifier(raw_df, clf_name, methods, metric='f1_macro'))
sig_df = pd.DataFrame(sig_rows)
display(sig_df)

print('Pairwise post-hoc vs GWO for performance (f1_macro)')
pairwise_rows = []
for clf_name in sorted(raw_df['classifier'].astype(str).unique()):
    pairwise_rows.extend(pairwise_vs_reference_for_classifier(raw_df, clf_name, methods, reference='GWO', metric='f1_macro'))
pairwise_df = pd.DataFrame(pairwise_rows)
display(pairwise_df)

print('Global significance for feature-selection time (fs_time_s)')
sig_df_time = pd.DataFrame([global_significance_for_fs(fs_df, methods, metric='fs_time_s')])
display(sig_df_time)

print('Pairwise post-hoc vs GWO for feature-selection time (fs_time_s)')
pairwise_time_df = pd.DataFrame(pairwise_vs_reference_for_fs(fs_df, methods, reference='GWO', metric='fs_time_s'))
display(pairwise_time_df)

print('Global significance for number of selected features (n_selected)')
sig_df_n_selected = pd.DataFrame([global_significance_for_fs(fs_df, methods, metric='n_selected')])
display(sig_df_n_selected)

print('Pairwise post-hoc vs GWO for number of selected features (n_selected)')
pairwise_n_selected_df = pd.DataFrame(pairwise_vs_reference_for_fs(fs_df, methods, reference='GWO', metric='n_selected'))
display(pairwise_n_selected_df)

print('Stability summary (mean pairwise Jaccard across runs)')
display(stability_df)

Global significance for performance (f1_macro)


,Classifier,metric,test,p_value,methods,n_runs
0,DecisionTree,f1_macro,Friedman,1.767445e-07,"[GWO, BA, LightGBM, Boruta]",10
1,GaussianNB,f1_macro,Friedman,1.526142e-06,"[GWO, BA, LightGBM, Boruta]",10
2,HistGradientBoosting,f1_macro,Friedman,4.759058e-05,"[GWO, BA, LightGBM, Boruta]",10
3,LogisticRegression,f1_macro,Friedman,3.772312e-07,"[GWO, BA, LightGBM, Boruta]",10
4,MLP,f1_macro,Friedman,1.217218e-06,"[GWO, BA, LightGBM, Boruta]",10
5,RandomForest,f1_macro,Friedman,1.172166e-06,"[GWO, BA, LightGBM, Boruta]",10


Pairwise post-hoc vs GWO for performance (f1_macro)


,Classifier,metric,reference,comparison,n_runs,mean_ref,mean_cmp,wins_ref,ties,wins_cmp,test,p_value,p_holm,better_by_mean
0,DecisionTree,f1_macro,GWO,BA,10,0.937937,0.906709,10,0,0,Wilcoxon,0.001953,0.007812,GWO
2,DecisionTree,f1_macro,GWO,LightGBM,10,0.937937,0.928889,10,0,0,Wilcoxon,0.001953,0.007812,GWO
3,DecisionTree,f1_macro,GWO,Boruta,10,0.937937,0.947976,0,0,10,Wilcoxon,0.001953,0.007812,Boruta
4,GaussianNB,f1_macro,GWO,BA,10,0.529187,0.454936,8,0,2,Wilcoxon,0.019531,0.039062,GWO
6,GaussianNB,f1_macro,GWO,LightGBM,10,0.529187,0.343243,10,0,0,Wilcoxon,0.001953,0.007812,GWO
7,GaussianNB,f1_macro,GWO,Boruta,10,0.529187,0.618239,0,0,10,Wilcoxon,0.001953,0.007812,Boruta
8,HistGradientBoosting,f1_macro,GWO,BA,10,0.923381,0.898003,10,0,0,Wilcoxon,0.001953,0.007812,GWO
10,HistGradientBoosting,f1_macro,GWO,LightGBM,10,0.923381,0.915406,10,0,0,Wilcoxon,0.001953,0.007812,GWO
11,HistGradientBoosting,f1_macro,GWO,Boruta,10,0.923381,0.929961,3,0,7,Wilcoxon,0.160156,0.320312,Boruta
12,LogisticRegression,f1_macro,GWO,BA,10,0.542956,0.505700,8,0,2,Wilcoxon,0.027344,0.027344,GWO


Global significance for feature-selection time (fs_time_s)


...

,metric,test,p_value,methods,n_runs
0,n_selected,Friedman,4.328423e-08,"[GWO, BA, LightGBM, Boruta]",10


Pairwise post-hoc vs GWO for number of selected features (n_selected)


,metric,reference,comparison,n_runs,mean_ref,mean_cmp,wins_ref,ties,wins_cmp,test,p_value,p_holm,faster_by_mean
0,n_selected,GWO,BA,10,10.0,10.0,0,10,0,Wilcoxon,1.000000,1.000000,GWO
2,n_selected,GWO,LightGBM,10,10.0,10.0,0,10,0,Wilcoxon,1.000000,1.000000,GWO
3,n_selected,GWO,Boruta,10,10.0,41.2,10,0,0,Wilcoxon,0.001953,0.007812,GWO


Stability summary (mean pairwise Jaccard across runs)


,method,mean_pairwise_jaccard,std_pairwise_jaccard,n_pairs
0,Boruta,0.969261,0.017814,45
1,LightGBM,0.927946,0.095033,45
3,GWO,0.248921,0.101118,45
4,BA,0.123380,0.068218,45


## Without FS 


In [4]:
import random
import time
import tracemalloc
from itertools import combinations
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

N_RUNS = 10
RUN_SEEDS = list(range(N_RUNS))
K_SUP = 10
N_FEATURES_TOTAL = int(X.shape[1])

FS_MAX_SAMPLES = 50000
FS_EPOCHS = 30
FS_POP_SIZE = 10
FS_CV_FOLDS = 3

FS_FITNESS_MODEL = 'DecisionTree'
FS_FITNESS_SCORING = 'f1_macro'

EVAL_CV_FOLDS = 3

def _subsample_for_fs_local(X_in: np.ndarray, y_in: np.ndarray, max_samples: int, seed: int):
    if (max_samples is None) or (X_in.shape[0] <= int(max_samples)):
        return X_in, y_in
    rng = np.random.default_rng(int(seed))
    idx = rng.choice(X_in.shape[0], size=int(max_samples), replace=False)
    return X_in[idx], y_in[idx]

def evaluate_cv_eff(clf, X_sub: np.ndarray, y_sub: np.ndarray, cv_folds: int, seed: int):
    cvk = StratifiedKFold(n_splits=int(cv_folds), shuffle=True, random_state=int(seed))

    accs, precs, recs, f1s, mccs = [], [], [], [], []
    train_time_s = 0.0
    infer_time_ms = 0.0

    tracemalloc.start()
    for tr, te in cvk.split(X_sub, y_sub):
        model = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler(with_mean=True)),
            ('clf', clone(clf))
        ])

        t0 = time.perf_counter()
        model.fit(X_sub[tr], y_sub[tr])
        train_time_s += (time.perf_counter() - t0)

        t1 = time.perf_counter()
        p = model.predict(X_sub[te])
        infer_time_ms += (time.perf_counter() - t1) * 1000.0

        accs.append(accuracy_score(y_sub[te], p))
        precs.append(precision_score(y_sub[te], p, average='macro', zero_division=0))
        recs.append(recall_score(y_sub[te], p, average='macro', zero_division=0))
        f1s.append(f1_score(y_sub[te], p, average='macro', zero_division=0))
        mccs.append(matthews_corrcoef(y_sub[te], p))

    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return {
        'accuracy': float(np.mean(accs)),
        'precision_macro': float(np.mean(precs)),
        'recall_macro': float(np.mean(recs)),
        'f1_macro': float(np.mean(f1s)),
        'mcc': float(np.mean(mccs)),
        'train_time_s': float(train_time_s),
        'inference_time_ms': float(infer_time_ms),
        'peak_mem_mb': float(peak) / (1024.0 * 1024.0)
    }

methods = ['Without FS']

clf_factories = {
    'RandomForest': lambda s: RandomForestClassifier(
        n_estimators=800,
        max_depth=30,
        min_samples_split=3,
        min_samples_leaf=1,
        max_features='sqrt',
        class_weight='balanced',
        max_samples=0.8,
        n_jobs=-1,
        random_state=42
    ),
    'DecisionTree': lambda s: DecisionTreeClassifier(random_state=int(s), class_weight='balanced'),
    'LogisticRegression': lambda s: LogisticRegression(max_iter=1000, class_weight='balanced', random_state=int(s)),
    'GaussianNB': lambda s: GaussianNB(),
    'HistGradientBoosting': lambda s: HistGradientBoostingClassifier(random_state=int(s), learning_rate=0.1),
    'MLP': lambda s: MLPClassifier(
        random_state=int(s),
        hidden_layer_sizes=(128, 64),
        max_iter=200,
        early_stopping=True,
        n_iter_no_change=10,
        validation_fraction=0.1
    )
}

fs_cache = {}

def make_fs_base(seed: int):
    if str(FS_FITNESS_MODEL).lower() in ['rf', 'randomforest', 'random_forest']:
        return RandomForestClassifier(
            n_estimators=120,
            max_depth=18,
            min_samples_split=4,
            min_samples_leaf=2,
            max_features='sqrt',
            bootstrap=True,
            max_samples=0.7,
            class_weight='balanced_subsample',
            n_jobs=-1,
            random_state=int(seed)
        )
    return DecisionTreeClassifier(random_state=int(seed), class_weight='balanced')

def _serialize_idx(idx):
    return ','.join(str(int(i)) for i in idx)

def _mean_pairwise_jaccard(index_sets):
    sets = [set(int(i) for i in s) for s in index_sets if len(s) > 0]
    if len(sets) < 2:
        return np.nan, np.nan, 0
    vals = []
    for a, b in combinations(sets, 2):
        union = len(a | b)
        vals.append(len(a & b) / union if union else 1.0)
    return float(np.mean(vals)), float(np.std(vals)), int(len(vals))

def get_idx_and_time(seed: int, method: str, X_fs: np.ndarray, y_fs: np.ndarray):
    key = (int(seed), str(method))
    if key in fs_cache:
        return fs_cache[key]

    random.seed(int(seed))
    np.random.seed(int(seed))

    base = make_fs_base(int(seed))
    t0 = time.perf_counter()
    if str(method) == 'Without FS':
        idx, _ = select_features_without_fs(X_fs, y_fs, clf=base, cv=FS_CV_FOLDS, scoring=str(FS_FITNESS_SCORING))
    else:
        raise ValueError(f'Unknown method: {method}')

    fs_time_s = float(time.perf_counter() - t0)
    fs_cache[key] = (idx, fs_time_s)
    return idx, fs_time_s

rows = []
for seed in RUN_SEEDS:
    X_fs, y_fs = _subsample_for_fs_local(X, y, FS_MAX_SAMPLES, seed)
    for method in methods:
        idx, fs_time_s = get_idx_and_time(seed, method, X_fs, y_fs)
        X_sub = X[:, idx]
        for clf_name, make_clf in clf_factories.items():
            m = evaluate_cv_eff(make_clf(seed), X_sub, y, cv_folds=EVAL_CV_FOLDS, seed=seed)
            rows.append({
                'seed': int(seed),
                'method': str(method),
                'classifier': str(clf_name),
                'k_target': int(K_SUP) if method in [] else np.nan,
                'n_selected': int(len(idx)),
                'selected_features': _serialize_idx(idx),
                'fs_time_s': float(fs_time_s),
                **m
            })

fs_rows = []
for (s, m), (idx, fs_time_s) in fs_cache.items():
    fs_rows.append({
        'seed': int(s),
        'method': str(m),
        'k_target': int(K_SUP) if str(m) in [] else np.nan,
        'n_selected': int(len(idx)),
        'selected_features': _serialize_idx(idx),
        'fs_time_s': float(fs_time_s)
    })
fs_df = pd.DataFrame(fs_rows)
fs_df = fs_df.sort_values(['seed', 'method']).reset_index(drop=True)
display(fs_df)
display(fs_df.groupby('method', as_index=False)[['fs_time_s', 'n_selected']].agg(['mean', 'std']).reset_index())

stability_rows = []
for method_name, sub_df in fs_df.groupby('method'):
    sets = []
    for txt in sub_df['selected_features'].astype(str):
        sets.append([int(x) for x in txt.split(',') if str(x).strip() != ''])
    mean_j, std_j, n_pairs = _mean_pairwise_jaccard(sets)
    stability_rows.append({
        'method': str(method_name),
        'mean_pairwise_jaccard': mean_j,
        'std_pairwise_jaccard': std_j,
        'n_pairs': n_pairs
    })
stability_df = pd.DataFrame(stability_rows).sort_values('mean_pairwise_jaccard', ascending=False).reset_index(drop=True)
print('Feature-selection stability across runs')
display(stability_df)

raw_df = pd.DataFrame(rows)
display(raw_df)
print('Rows:', len(raw_df))

metrics_cols = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'mcc', 'fs_time_s', 'train_time_s', 'inference_time_ms', 'peak_mem_mb', 'n_selected']

means = raw_df.groupby(['method', 'classifier'])[metrics_cols].mean().reset_index()
means = means.rename(columns={
    'method': 'Algorithm',
    'classifier': 'Classifier',
    'accuracy': 'Accuracy',
    'precision_macro': 'Precision',
    'recall_macro': 'Recall',
    'f1_macro': 'F1_macro',
    'mcc': 'MCC',
    'n_selected': 'N_Selected'
})

eval_df = means[['Algorithm', 'Classifier', 'Accuracy', 'Precision', 'Recall', 'F1_macro', 'MCC', 'N_Selected', 'fs_time_s', 'train_time_s', 'inference_time_ms', 'peak_mem_mb']].copy()
eval_df = eval_df.sort_values(['F1_macro', 'MCC', 'Accuracy'], ascending=[False, False, False]).reset_index(drop=True)
display(eval_df)

print('Deployment-oriented summary (lower downstream cost is better)')
deployment_df = eval_df[['Algorithm', 'Classifier', 'N_Selected', 'train_time_s', 'inference_time_ms', 'peak_mem_mb']].copy()
deployment_df = deployment_df.sort_values(['N_Selected', 'peak_mem_mb', 'inference_time_ms', 'train_time_s'], ascending=[True, True, True, True]).reset_index(drop=True)
display(deployment_df)

agg = raw_df.groupby(['method', 'classifier'])[metrics_cols].agg(['mean', 'std'])
agg.columns = [f"{m}_{s}" for (m, s) in agg.columns]
agg = agg.reset_index()

def fmt_pair(m, s, is_time=False):
    if pd.isna(m):
        return ''
    if pd.isna(s):
        return f"{m:.4f}" if not is_time else f"{m:.3f}"
    return (f"{m:.4f} +- {s:.4f}" if not is_time else f"{m:.3f} +- {s:.3f}")

final_table = pd.DataFrame({
    'Method': agg['method'].astype(str),
    'Classifier': agg['classifier'].astype(str),
})

for src, dst in {
    'accuracy': 'accuracy',
    'precision_macro': 'precision_macro',
    'recall_macro': 'recall_macro',
    'f1_macro': 'f1_macro',
    'mcc': 'mcc',
    'n_selected': 'n_selected',
    'fs_time_s': 'fs_time_s',
    'train_time_s': 'train_time_s',
    'inference_time_ms': 'inference_time_ms',
    'peak_mem_mb': 'peak_mem_mb'
}.items():
    mcol = f"{src}_mean"
    scol = f"{src}_std"
    is_time = src in ['fs_time_s', 'train_time_s', 'inference_time_ms', 'peak_mem_mb']
    final_table[dst] = [fmt_pair(m, s, is_time=is_time) for m, s in zip(agg[mcol], agg[scol])]

final_table = final_table.sort_values(['Classifier', 'Method']).reset_index(drop=True)
display(final_table)

,seed,method,k_target,n_selected,selected_features,fs_time_s
0,0,Without FS,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.929884
1,1,Without FS,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.836359
2,2,Without FS,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.947452
3,3,Without FS,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.933598
4,4,Without FS,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.950988
5,5,Without FS,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.758935
6,6,Without FS,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",2.004424
7,7,Without FS,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.932611
8,8,Without FS,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.864838
9,9,Without FS,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.952452


index      method fs_time_s           n_selected     
                         mean       std       mean  std
0     0  Without FS  1.911154  0.071129       51.0  0.0

Feature-selection stability across runs


,method,mean_pairwise_jaccard,std_pairwise_jaccard,n_pairs
0,Without FS,1.0,0.0,45


,seed,method,classifier,k_target,n_selected,selected_features,fs_time_s,accuracy,precision_macro,recall_macro,f1_macro,mcc,train_time_s,inference_time_ms,peak_mem_mb
0,0,Without FS,RandomForest,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.929884,0.959928,0.968909,0.931891,0.949517,0.934710,249.451818,8387.550500,779.756327
1,0,Without FS,DecisionTree,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.929884,0.962322,0.971591,0.924790,0.947068,0.938656,36.403723,441.553563,779.554370
2,0,Without FS,LogisticRegression,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.929884,0.868607,0.718194,0.824532,0.751872,0.791711,366.474471,704.423522,779.552401
3,0,Without FS,GaussianNB,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.929884,0.778467,0.631963,0.617641,0.599059,0.647655,8.423856,1376.084547,779.545933
4,0,Without FS,HistGradientBoosting,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.929884,0.945683,0.924071,0.890753,0.902975,0.911629,79.777777,4905.024719,779.641213
5,0,Without FS,MLP,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.929884,0.944441,0.947243,0.883133,0.912693,0.909277,654.772901,957.965766,779.561975
6,1,Without FS,RandomForest,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.836359,0.960001,0.969379,0.931691,0.949633,0.934832,258.811721,7583.350242,779.610837
7,1,Without FS,DecisionTree,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.836359,0.962662,0.972433,0.924915,0.947527,0.939223,37.103135,427.053935,779.552803
8,1,Without FS,LogisticRegression,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.836359,0.868778,0.718932,0.825030,0.752816,0.791941,362.769937,484.537416,779.550624
9,1,Without FS,GaussianNB,NaN,51,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50",1.836359,0.778066,0.631795,0.617254,0.598522,0.646993,8.766534,1212.404043,779.544991


Rows: 60


,Algorithm,Classifier,Accuracy,Precision,Recall,F1_macro,MCC,N_Selected,fs_time_s,train_time_s,inference_time_ms,peak_mem_mb
0,Without FS,RandomForest,0.960021,0.968849,0.931952,0.949530,0.934867,51.0,1.911154,258.069956,8571.217914,779.620582
1,Without FS,DecisionTree,0.962749,0.972561,0.925650,0.947979,0.939366,51.0,1.911154,35.911660,440.237171,779.551052
2,Without FS,HistGradientBoosting,0.952194,0.955416,0.905553,0.928150,0.922286,51.0,1.911154,77.189112,4698.291523,779.611825
3,Without FS,MLP,0.944338,0.945834,0.884746,0.913144,0.909104,51.0,1.911154,704.550830,886.553415,779.554934
4,Without FS,LogisticRegression,0.868783,0.718786,0.824896,0.752603,0.791952,51.0,1.911154,363.861198,511.244103,779.550471
5,Without FS,GaussianNB,0.778141,0.631725,0.617064,0.598237,0.647121,51.0,1.911154,8.266062,1219.398344,779.545019


Deployment-oriented summary (lower downstream cost is better)


,Algorithm,Classifier,N_Selected,train_time_s,inference_time_ms,peak_mem_mb
0,Without FS,GaussianNB,51.0,8.266062,1219.398344,779.545019
1,Without FS,LogisticRegression,51.0,363.861198,511.244103,779.550471
2,Without FS,DecisionTree,51.0,35.911660,440.237171,779.551052
3,Without FS,MLP,51.0,704.550830,886.553415,779.554934
4,Without FS,HistGradientBoosting,51.0,77.189112,4698.291523,779.611825
5,Without FS,RandomForest,51.0,258.069956,8571.217914,779.620582


,Method,Classifier,accuracy,precision_macro,recall_macro,f1_macro,mcc,n_selected,fs_time_s,train_time_s,inference_time_ms,peak_mem_mb
0,Without FS,DecisionTree,0.9627 +- 0.0002,0.9726 +- 0.0006,0.9256 +- 0.0007,0.9480 +- 0.0006,0.9394 +- 0.0004,51.0000 +- 0.0000,1.911 +- 0.071,35.912 +- 0.623,440.237 +- 43.722,779.551 +- 0.003
1,Without FS,GaussianNB,0.7781 +- 0.0003,0.6317 +- 0.0005,0.6171 +- 0.0005,0.5982 +- 0.0008,0.6471 +- 0.0005,51.0000 +- 0.0000,1.911 +- 0.071,8.266 +- 0.310,1219.398 +- 77.677,779.545 +- 0.001
2,Without FS,HistGradientBoosting,0.9522 +- 0.0034,0.9554 +- 0.0188,0.9056 +- 0.0083,0.9282 +- 0.0143,0.9223 +- 0.0056,51.0000 +- 0.0000,1.911 +- 0.071,77.189 +- 9.189,4698.292 +- 1253.186,779.612 +- 0.012
3,Without FS,LogisticRegression,0.8688 +- 0.0002,0.7188 +- 0.0004,0.8249 +- 0.0003,0.7526 +- 0.0005,0.7920 +- 0.0003,51.0000 +- 0.0000,1.911 +- 0.071,363.861 +- 3.400,511.244 +- 97.424,779.550 +- 0.002
4,Without FS,MLP,0.9443 +- 0.0006,0.9458 +- 0.0028,0.8847 +- 0.0022,0.9131 +- 0.0015,0.9091 +- 0.0010,51.0000 +- 0.0000,1.911 +- 0.071,704.551 +- 137.992,886.553 +- 128.343,779.555 +- 0.003
5,Without FS,RandomForest,0.9600 +- 0.0001,0.9688 +- 0.0003,0.9320 +- 0.0002,0.9495 +- 0.0002,0.9349 +- 0.0001,51.0000 +- 0.0000,1.911 +- 0.071,258.070 +- 3.689,8571.218 +- 1158.646,779.621 +- 0.049
